# 🧠 AI Chatbot for FAQs - Evaluation & Core Algorithms

This notebook covers:
1. Ingesting and tokenizing FAQ documents
2. Building TF-IDF similarity vectors
3. Running accuracy & keyword-overlap evaluations
4. Optional LangChain / Generative AI integration

In [ ]:
import os
import json
from faq_engine import FAQEngine

print("✅ Setup complete!")

## 1. Load Knowledge Base & Test Ingestion

In [ ]:
engine = FAQEngine("faq_documents")
stats = engine.get_stats()
print(f"Total Chunks Loaded: {stats['total_chunks']}")
for doc in stats['documents']:
    print(f" - {doc['name']}: {doc['chunks']} chunks")

## 2. Accuracy & Evaluation Benchmark

In [ ]:
# Benchmark test suite mapping sample queries to expected keywords in response
test_suite = [
    {
        "query": "What are your support hours?",
        "expected_keywords": ["8:00 AM", "6:00 PM", "PST", "Standard Support"]
    },
    {
        "query": "How much does CloudSync Pro cost?",
        "expected_keywords": ["$49", "$149", "$499", "Starter"]
    },
    {
        "query": "How do I reset my password?",
        "expected_keywords": ["Forgot Password", "email", "reset"]
    },
    {
        "query": "What is your refund policy?",
        "expected_keywords": ["30-day", "money-back", "refund"]
    },
    {
        "query": "Are you GDPR compliant?",
        "expected_keywords": ["GDPR", "CCPA", "privacy"]
    }
]

passed = 0
print("🔬 Running Evaluation Benchmark...\n")
for test in test_suite:
    result = engine.ask(test["query"])
    answer_text = result["answer"].lower()
    matched_kw = [kw for kw in test["expected_keywords"] if kw.lower() in answer_text]
    score = len(matched_kw) / len(test["expected_keywords"])
    
    is_pass = score >= 0.5
    if is_pass: passed += 1
    status = "✅ PASS" if is_pass else "❌ FAIL"
    
    print(f"{status} | Query: '{test['query']}'")
    print(f"      Confidence: {result['confidence']}% | Source: {result['sources']}")
    print(f"      Keywords Found: {matched_kw} ({int(score * 100)}% match)\n")

accuracy = (passed / len(test_suite)) * 100
print(f"🎯 Overall Benchmark Accuracy: {accuracy:.1f}% ({passed}/{len(test_suite)} tests passed)")

## 3. Dynamic Document Addition & Instant Indexing

In [ ]:
custom_faq = """
## Q: Do you provide free migration assistance?
A: Yes, all Enterprise customers receive free migration assistance including dedicated database engineers and automated data replication tools.
"""

new_chunks = engine.add_document(custom_faq, "custom_faq.txt")
print(f"Added {new_chunks} new chunk(s)!")

# Test newly added chunk immediately
res = engine.ask("Do you help with database migration?")
print("\nQuery: 'Do you help with database migration?'")
print(f"Confidence: {res['confidence']}%")
print(f"Answer: {res['answer']}")